# Results 3 — Systematic colocalisation and effector-gene prediction

What supports each L2G gene assignment, and how that support changed over time.

The model-evaluation numbers of this subsection (average precision 0.81, area under the curve
0.95, recall 0.65, the false discovery rates of Supplementary Table 12, and the comparison
against the previous Open Targets model) need the L2G training set and the saved held-out
split, neither of which is available; see GAPS.md. The same applies to the loss-of-function
constraint enrichment (OR = 1.9) and to the secondary-signal analysis of Supplementary
Results 5.

Also writes the Extended Data Figure 5 Venn counts and the Extended Data Figure 6 temporal
table.

In [1]:
import pandas as pd
from gentropy.common.session import Session
from pyspark.sql import functions as f

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
numbers = {}

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/19 23:49:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Every protein-coding gene scored against a qualifying credible set.
qualifying = (
    session.spark.read.parquet(paper.derived("qualifying_credible_sets"))
    .select("studyLocusId")
    .union(session.spark.read.parquet(paper.derived("qualifying_measurement_credible_sets")).select("studyLocusId"))
    .distinct()
)
scored = (
    session.spark.read.parquet(paper.release("l2g_feature_matrix"))
    .filter(f.col("isProteinCoding") == 1)
    .select("studyLocusId", "geneId")
    .join(qualifying, "studyLocusId", "inner")
)
numbers["R3.04"] = scored.count()
print("credible set to protein-coding gene pairs scored:", numbers["R3.04"])

credible set to protein-coding gene pairs scored: 7066749


## What supports each assignment

Over every gene prioritisation on a qualifying credible set, disease or measurement.

In [3]:
assignments = (
    session.spark.read.parquet(paper.derived("prioritised_genes_diseases"))
    .select("studyLocusId", "geneId", "eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS")
    .unionByName(
        session.spark.read.parquet(paper.derived("prioritised_genes_measurements")).select(
            "studyLocusId", "geneId", "eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS"
        )
    )
    .distinct()
    .toPandas()
)
total = len(assignments)
print("CS-gene prioritisations:", total)

numbers["R3.10"] = round(100 * assignments["eQTL_coloc"].mean(), 1)
numbers["R3.11"] = round(100 * assignments["pQTL_coloc"].mean(), 1)
numbers["R3.15"] = round(100 * assignments["distanceTSS"].mean(), 1)
numbers["R3.17"] = round(100 * assignments["VEP"].mean(), 1)

# "81.2% (424,781) genes were nearest to TSS, with 46.1% (241,404) having no PAV, eQTL or pQTL
# evidence": the published count 241,404 is the nearest-gene assignments carrying none of the
# three, and 46.1% is that count over **all** assignments, not over the nearest ones (which reads
# 56.8%). Settled against the counts printed in Supplementary Results 3.
unsupported = (assignments["VEP"] == 0) & (assignments["eQTL_coloc"] == 0) & (assignments["pQTL_coloc"] == 0)
nearest_unsupported = (assignments["distanceTSS"] == 1) & unsupported
numbers["R3.16"] = round(100 * nearest_unsupported.mean(), 1)
print({k: numbers[k] for k in ["R3.10", "R3.11", "R3.15", "R3.16", "R3.17"]})

CS-gene prioritisations: 523409
{'R3.10': np.float64(36.7), 'R3.11': np.float64(5.7), 'R3.15': np.float64(81.2), 'R3.16': np.float64(46.1), 'R3.17': np.float64(12.1)}


## Extended Data Figure 5 — the four reasons a gene can be prioritised

In [4]:
venn = (
    assignments.groupby(["eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS"])
    .size()
    .rename("assignments")
    .reset_index()
    .sort_values("assignments", ascending=False)
)
venn.to_csv(paper.derived("extended_figure_5_venn.csv"), index=False)
venn

,eQTL_coloc,pQTL_coloc,VEP,distanceTSS,assignments
1,0,0,0,1,241404
9,1,0,0,1,124809
0,0,0,0,0,40635
8,1,0,0,0,32636
3,0,0,1,1,24347
11,1,0,1,1,14982
2,0,0,1,0,8827
13,1,1,0,1,6638
10,1,0,1,0,5739
7,0,1,1,1,5433


## Extended Data Figure 6 — support over time

For each year, a gene counts as supported if any credible set published up to that year and
prioritising it carried a protein-altering variant or a molQTL colocalisation.

In [5]:
genes = (
    session.spark.read.parquet(paper.derived("prioritised_genes_diseases"))
    .select("geneId", "year", "score", "eQTL_coloc", "pQTL_coloc", "VEP")
    .toPandas()
)

records = []
for year in range(2006, 2025):
    subset = genes[genes["year"] <= year]
    if subset.empty:
        continue
    per_gene = subset.groupby("geneId").agg(
        maxScore=("score", "max"),
        pav=("VEP", "max"),
        eqtl=("eQTL_coloc", "max"),
        pqtl=("pQTL_coloc", "max"),
    )
    coloc = (per_gene["eqtl"] == 1) | (per_gene["pqtl"] == 1)
    pav = per_gene["pav"] == 1
    records.append(
        {
            "year": year,
            "genes": len(per_gene),
            "meanMaxL2G": float(per_gene["maxScore"].mean()),
            "seMaxL2G": float(per_gene["maxScore"].sem()),
            "pavOnly": int((pav & ~coloc).sum()),
            "pavAndColoc": int((pav & coloc).sum()),
            "colocOnly": int((~pav & coloc).sum()),
            "neither": int((~pav & ~coloc).sum()),
        }
    )

temporal = pd.DataFrame(records)
temporal["pctNeither"] = 100 * temporal["neither"] / temporal["genes"]
temporal.to_csv(paper.derived("extended_figure_6_temporal.csv"), index=False)

by_year = temporal.set_index("year")["pctNeither"]
numbers["R3.08"] = round(float(by_year.loc[2015]), 0)
numbers["R3.09"] = round(float(by_year.loc[2024]), 0)
print({k: numbers[k] for k in ["R3.08", "R3.09"]})
temporal[["year", "genes", "meanMaxL2G", "pctNeither"]].round(3).tail(12)

{'R3.08': 49.0, 'R3.09': 40.0}


,year,genes,meanMaxL2G,pctNeither
7,2013,996,0.608,46.787
8,2014,1152,0.603,48.698
9,2015,1332,0.603,48.724
10,2016,1525,0.608,49.377
11,2017,1923,0.613,47.842
12,2018,2629,0.616,46.596
13,2019,3145,0.620,46.550
14,2020,3781,0.624,46.152
15,2021,4789,0.634,44.101
16,2022,5494,0.635,43.939


## MST1, a non-nearest prioritisation

The inflammatory bowel disease credible set at 3:49676792:T/C, where L2G nominates the third
closest protein-coding gene.

In [6]:
mst1 = (
    session.spark.read.parquet(paper.derived("prioritised_genes_annotated"))
    .filter(f.col("variantId") == "3_49676792_T_C")
    .select("variantId", "studyId", "geneId", "score", "eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS")
    .toPandas()
)
mst1_rows = mst1[mst1["geneId"] == "ENSG00000173531"]  # MST1
if len(mst1_rows):
    numbers["R3.18"] = round(float(mst1_rows["score"].max()), 2)
print("MST1 L2G score:", numbers.get("R3.18"))
mst1.sort_values("score", ascending=False).head(10)

MST1 L2G score: 0.86


,variantId,studyId,geneId,score,eQTL_coloc,pQTL_coloc,VEP,distanceTSS
0,3_49676792_T_C,GCST004133,ENSG00000173531,0.861804,1,1,0,0
2,3_49676792_T_C,GCST004131,ENSG00000173531,0.779485,1,1,0,0
3,3_49676792_T_C,GCST90292538,ENSG00000173531,0.777470,1,1,1,0
1,3_49676792_T_C,GCST90479628,ENSG00000173531,0.673413,1,1,1,0


## Numbers

In [7]:
print(paper.save_results("colocalisation_l2g", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/colocalisation_l2g.json


,computed
R3.04,7066749.00
R3.10,36.70
R3.11,5.70
R3.15,81.20
R3.17,12.10
R3.16,46.10
R3.08,49.00
R3.09,40.00
R3.18,0.86
